# Iris Experiment: Random Forest Classifier

**Purpose**: Train and evaluate a Random Forest classifier on the Iris dataset, tracking all runs in MLflow.  
**Hypothesis (H2)**: A Random Forest classifier will achieve >95% test accuracy using all 4 features.  
**Author**: {your name}  
**Date**: {YYYY-MM-DD}  

**Prerequisites**: Complete `01_eda.ipynb` first.

## Setup

In [ ]:
import subprocess
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from datetime import date

# --- SEED: set before ANY random operation ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"SEED = {SEED}")

## Hyperparameters

All hyperparameters defined here — never hardcoded in the training block.

In [ ]:
# --- Hyperparameters (all logged to MLflow below) ---
N_ESTIMATORS = 100
MAX_DEPTH = None        # None = grow until leaves are pure
MIN_SAMPLES_SPLIT = 2
MIN_SAMPLES_LEAF = 1
TEST_SIZE = 0.2
VAL_CV_FOLDS = 5
DATASET = "iris_sklearn_builtin"
MODEL_TYPE = "RandomForestClassifier"
FEATURE_SET = "all_4_features"

HYPOTHESIS = "RF achieves >95% test accuracy using all 4 features"

## Load and Split Data

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Class distribution (train): {y_train.value_counts().to_dict()}")

## MLflow Run

In [ ]:
mlflow.set_experiment("iris-classification")

with mlflow.start_run() as run:
    # --- Tag: run name and hypothesis ---
    run_name = f"{MODEL_TYPE}_{DATASET}_{date.today().isoformat()}"
    mlflow.set_tag("mlflow.runName", run_name)
    mlflow.set_tag("hypothesis", HYPOTHESIS)

    # --- Tag: git commit for full reproducibility ---
    try:
        git_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"]
        ).decode().strip()
    except Exception:
        git_commit = "unavailable"
    mlflow.set_tag("git_commit", git_commit)

    # --- Log all hyperparameters ---
    mlflow.log_param("random_seed", SEED)
    mlflow.log_param("n_estimators", N_ESTIMATORS)
    mlflow.log_param("max_depth", str(MAX_DEPTH))
    mlflow.log_param("min_samples_split", MIN_SAMPLES_SPLIT)
    mlflow.log_param("min_samples_leaf", MIN_SAMPLES_LEAF)
    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("cv_folds", VAL_CV_FOLDS)
    mlflow.log_param("dataset", DATASET)
    mlflow.log_param("feature_set", FEATURE_SET)
    mlflow.log_param("model_type", MODEL_TYPE)

    # --- Train ---
    model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        random_state=SEED
    )
    model.fit(X_train, y_train)

    # --- Cross-validation (validation metric) ---
    cv_scores = cross_val_score(model, X_train, y_train, cv=VAL_CV_FOLDS, scoring='accuracy')
    for fold, score in enumerate(cv_scores):
        mlflow.log_metric("val_accuracy_fold", score, step=fold)
    mlflow.log_metric("val_accuracy_mean", cv_scores.mean())
    mlflow.log_metric("val_accuracy_std", cv_scores.std())

    # --- Train metrics ---
    train_preds = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, train_preds)
    train_f1 = f1_score(y_train, train_preds, average='macro')
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_f1_macro", train_f1)

    # --- Test metrics (run ONCE, only here) ---
    test_preds = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, test_preds)
    test_f1 = f1_score(y_test, test_preds, average='macro')
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_macro", test_f1)

    print(f"Train accuracy: {train_accuracy:.4f}")
    print(f"Val accuracy (CV mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"Test accuracy: {test_accuracy:.4f}")

    # --- Confusion matrix artifact ---
    import os
    os.makedirs("outputs", exist_ok=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_test, test_preds)
    sns_cm = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(iris.target_names)
    ax.set_yticklabels(iris.target_names)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix — Test Set')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
    plt.tight_layout()
    cm_path = "outputs/confusion_matrix.png"
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    mlflow.log_artifact(cm_path)
    plt.show()

    # --- Feature importance artifact ---
    fi = pd.Series(model.feature_importances_, index=iris.feature_names).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    fi.plot(kind='barh', ax=ax)
    ax.set_title('Feature Importances (Gini)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    fi_path = "outputs/feature_importance.png"
    plt.savefig(fi_path, dpi=150, bbox_inches='tight')
    mlflow.log_artifact(fi_path)
    plt.show()

    # --- Log model ---
    mlflow.sklearn.log_model(model, "model")

    run_id = run.info.run_id
    print(f"\nMLflow run ID: {run_id}")

## Classification Report

In [ ]:
print(classification_report(y_test, test_preds, target_names=iris.target_names))

## Hypothesis Verdict

**H2**: A Random Forest classifier will achieve >95% test accuracy using all 4 features.

Fill in after running:
- Test accuracy: `{test_accuracy:.4f}`
- Verdict: **{CONFIRMED / REJECTED}**
- Notes: {any unexpected findings}

---

If test accuracy ≥ 0.95 → proceed to model registration and model card.  
If test accuracy < 0.95 → diagnose (check confusion matrix, try different features or algorithm).

## Register Winning Model (run only when satisfied with results)

In [ ]:
# Uncomment after confirming test results meet acceptance criteria

# model_uri = f"runs:/{run_id}/model"
# registered = mlflow.register_model(model_uri, "IrisClassifier")

# client = mlflow.tracking.MlflowClient()
# client.set_registered_model_alias(
#     "IrisClassifier",
#     "champion",
#     version=registered.version
# )
# print(f"Registered model version {registered.version} as 'champion'")